In [1]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, VotingRegressor, BaggingRegressor, StackingRegressor
from sklearn.model_selection import GridSearchCV, cross_validate, train_test_split
from sklearn.metrics import root_mean_squared_error
import itertools
import joblib
import requests
import json

# Research


## Forecast

### 1. Data Preparation

In [2]:
df = pd.read_csv("data/epi_r.csv")

In [3]:
df.drop(columns=['title', 'calories', 'protein', 'fat', 'sodium', '#cakeweek', '#wasteless', '22-minute meals', '3-ingredient recipes', '30 days of groceries',
                 'advance prep required', 'alabama', 'alaska', 'alcoholic', 'anniversary', 'anthony bourdain', 'aperitif', 'aperitif', 'arizona', 'aspen', 'atlanta', 'australia',
                 'back to school', 'backyard bbq', 'bastille day', 'beverly hills', 'birthday', 'blender', 'boil', 'bon appétit', 'bon app��tit', 'boston', 'braise', 'breakfast', 'broil', 'brooklyn', 'brunch', 'buffalo', 'buffet', 'bulgaria',
                 'california', 'cambridge', 'camping', 'canada', 'candy thermometer', 'chicago', 'chill', 'christmas', 'christmas eve', 'cocktail', 'cocktail party', 'coffee grinder', 'colorado', 'columbus', 'connecticut', 'cook like a diner', 'cookbook critic', 'costa mesa', 'cuba',
                 'dairy free', 'dallas', 'deep-fry', 'denver', 'dessert', 'digestif', 'dinner', 'diwali', 'dominican republic', 'dorie greenspan', 'double boiler', 'drink', 'drinks',
                 'easter', 'edible gift', 'egypt', 'emeril lagasse', 'engagement party', 'england', 'entertaining', 'epi + ushg', 'epi loves the microwave',
                 'fall', 'family reunion', 'fat free', "father's day", 'flaming hot summer', 'florida', 'food processor', 'fourth of july', 'france', 'frankenrecipe', 'freeze/chill', 'freezer food', 'friendsgiving', 'frozen dessert', 'fry', 'fruit',
                 'game', 'georgia', 'germany', 'gourmet', 'graduation', 'grill', 'grill/barbecue', 'guam',
                 'haiti', 'halloween', 'hanukkah', 'hawaii', 'healdsburg', 'healthy', 'high fiber', 'hollywood', "hors d'oeuvre", 'hot drink', 'house & garden', 'house cocktail', 'houston',
                 'ice cream machine', 'idaho', 'illinois', 'indiana', 'iowa', 'ireland', 'israel', 'italy',
                 'jamaica', 'japan', 'juicer',
                 'kansas', 'kansas city', 'kentucky', 'kentucky derby', 'kid-friendly', 'kidney friendly', 'kitchen olympics', 'kosher', 'kosher for passover',
                 'labor day', 'lancaster', 'las vegas', 'london', 'los angeles', 'louisiana', 'louisville', 'low cal', 'low carb', 'low cholesterol', 'low fat', 'low sodium', 'low sugar', 'low/no sugar', 'lunar new year', 'lunch',
                 'maine', 'maryland', 'massachusetts', 'mexico', 'miami', 'michigan', 'microwave', 'minneapolis', 'minnesota', 'mississippi', 'missouri', 'mixer', 'monterey jack', 'mortar and pestle', "mother's day",
                 'nancy silverton', 'nebraska', 'new hampshire', 'new jersey', 'new mexico', 'new orleans', "new year's day", "new year's eve", 'new york', 'no meat, no problem', 'no sugar added', 'no-cook', 'non-alcoholic', 'north carolina',
                 'ohio', 'oklahoma', 'oktoberfest', 'one-pot meal', 'oregon', 'organic',
                 'pan-fry', 'parade', 'paris', 'party', 'pasadena', 'passover', 'pasta maker', 'peanut free', 'pennsylvania', 'persian new year', 'peru', 'pescatarian', 'philippines', 'pittsburgh', 'pizza', 'poach', 'poker/game night', 'port', 'portland', 'pot pie', 'potluck', 'pressure cooker', 'providence', 'purim',
                 'quick & easy', 'quick and healthy',
                 'ramadan', 'raw', 'rhode island', 'roast', 'rosh hashanah/yom kippur', 'rub',
                 'san francisco', 'sandwich', 'sandwich theory', 'santa monica', 'seattle', 'self', 'shower', 'side', 'simmer', 'skewer', 'slow cooker', 'smoker', 'smoothie', 'soup/stew', 'south carolina', 'soy free', 'spain', 'spring', 'st. louis', "st. patrick's day", 'steam', 'stew', 'stir-fry', 'stock', 'stuffing/dressing', 'sugar conscious', 'summer', 'super bowl', 'suzanne goin', 'switzerland',
                 'tailgating', 'tennessee', 'tested & improved', 'texas', 'thanksgiving', 'tree nut free',
                 'utah', "valentine's day", 'vegan', 'vegetarian', 'vermont', 'virginia',
                 'washington', 'washington, d.c.', 'wedding', 'west virginia', 'wheat/gluten-free', 'whole wheat', 'windsor', 'winter', 'wisconsin', 'wok',
                 'cookbooks', 'leftovers', 'snack', 'snack week' ],
        inplace=True)

In [4]:
X = df.drop(columns = ['rating'])
y = df['rating']

In [5]:
X = X.astype('int8')

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21)

In [7]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 20052 entries, 0 to 20051
Columns: 403 entries, almond to turkey
dtypes: int8(403)
memory usage: 7.7 MB


### 2.Regression

In [8]:
n_jobs = 2
cv =5

#### Regression Algorithms

In [9]:
lin_reg = LinearRegression()
lin_score = cross_validate(lin_reg, X_train, y_train, return_train_score=True, cv = cv, scoring='neg_root_mean_squared_error')
print(f"Average RMSE on crossval is {-lin_score['test_score'].mean()}")

Average RMSE on crossval is 1.284830843430915


In [10]:
tree_reg_params = {
    'max_depth' : [2, 3, 5, 6, 7, 10, 15, 20]
}
tree_reg = DecisionTreeRegressor(random_state=21)
tree_reg_gs = GridSearchCV(tree_reg, tree_reg_params, n_jobs = n_jobs, cv = cv, scoring='neg_root_mean_squared_error')
tree_reg_gs.fit(X_train, y_train)

print(f"Best parameters are {tree_reg_gs.best_params_}")
print(f"Average RMSE on crossval for best parameters is {-tree_reg_gs.best_score_}")

Best parameters are {'max_depth': 5}
Average RMSE on crossval for best parameters is 1.3023807499251423


In [11]:
rf_reg_params = {
    'max_depth' : [10, 20, 23, 40],
    'n_estimators' : [10, 25, 50, 75] 
}
rf_reg = RandomForestRegressor(random_state = 21)
rf_reg_gs = GridSearchCV(rf_reg, rf_reg_params, n_jobs = n_jobs, cv = cv, scoring='neg_root_mean_squared_error')
rf_reg_gs.fit(X_train, y_train)

print(f"Best parameters are {rf_reg_gs.best_params_}")
print(f"Average RMSE on crossval for best parameters is {-rf_reg_gs.best_score_}")

Best parameters are {'max_depth': 40, 'n_estimators': 75}
Average RMSE on crossval for best parameters is 1.2720710578642898


In [12]:
best_reg_score = root_mean_squared_error(y_test, rf_reg_gs.best_estimator_.predict(X_test))
print(f"Best RMSE-score on test subsample for best regression model is {best_reg_score}")

Best RMSE-score on test subsample for best regression model is 1.2512678209998125


#### Regression Ensembles

In [13]:
voting_reg_gs_params = {'weights' : list(itertools.product(
    range(1,6),
    range(1,6),
    range(1,6)))}
voting_reg = VotingRegressor(estimators=[('lin', lin_reg), ('tree', tree_reg_gs.best_estimator_), ('rf', rf_reg_gs.best_estimator_)])
voting_reg_gs = GridSearchCV(voting_reg, voting_reg_gs_params, n_jobs = n_jobs, cv = cv, scoring='neg_root_mean_squared_error')
voting_reg_gs.fit(X_train, y_train)
print(f"Best parameters are {voting_reg_gs.best_params_}")
print(f"Average RMSE on crossval for best parameters is {-voting_reg_gs.best_score_}")

KeyboardInterrupt: 

In [ ]:
bagging_reg_gs_params = {
    'n_estimators' : [10, 20, 30, 50]
}
bagging_reg = BaggingRegressor(estimator = lin_reg, random_state = 21)
bagging_reg_gs = GridSearchCV(bagging_reg, bagging_reg_gs_params, n_jobs = n_jobs, cv = cv, scoring='neg_root_mean_squared_error')
bagging_reg_gs.fit(X_train, y_train)
print(f"Best parameters are {bagging_reg_gs.best_params_}")
print(f"Average RMSE on crossval for best parameters is {-bagging_reg_gs.best_score_}")

Best parameters are {'n_estimators': 50}
Average RMSE on crossval for best parameters is 1.281703671975928


In [ ]:
stacking_reg_gs_params = {
    'passthrough' : (True, False)
}
stacking_reg = StackingRegressor(estimators=[('lin', lin_reg), ('tree', tree_reg_gs.best_estimator_), ('rf', rf_reg_gs.best_estimator_)])
stacking_reg_gs = GridSearchCV(stacking_reg, stacking_reg_gs_params, n_jobs = n_jobs, cv = cv, scoring='neg_root_mean_squared_error')
stacking_reg_gs.fit(X_train, y_train)
print(f"Best parameters are {stacking_reg_gs.best_params_}")
print(f"Average RMSE on crossval for best parameters is {-stacking_reg_gs.best_score_}")

Best parameters are {'passthrough': False}
Average RMSE on crossval for best parameters is 1.2574210751200585


In [ ]:
best_ensemble_score = root_mean_squared_error(y_test, voting_reg_gs.best_estimator_.predict(X_test))
print(f"Best RMSE-score on test subsample for best ensemble regression model is {best_ensemble_score}")

Best RMSE-score on test subsample for best ensemble regression model is 1.2298765545823163


#### Naive regressor

In [ ]:
y_naive_pred = [y_test.mean()] * len(y_test)
naive_rmse = root_mean_squared_error(y_test, y_naive_pred)
print(f"RMSE-score for naive regressor is {naive_rmse}")

RMSE-score for naive regressor is 1.3072126305093743


### 3. Classification

### 4. Decision

## Nutrition Facts

In [ ]:
api_key = 'PGRM6CVqVJAO7UGSTTsELnZhptWbIU0crzwl4Pyh'
api_url = f"https://api.nal.usda.gov/fdc/v1/foods/search?api_key={api_key}&query="

In [ ]:
rows = []
for col in X.columns:
    row = {'title' : col}
    query = col.replace(' ', '%20').replace('/', '%20')
    response = requests.get(api_url + query)
    py_dict = json.loads(response.text)
    if py_dict['totalHits'] > 0:
        for nutrient in py_dict['foods'][0]['foodNutrients']:
            nutrientName = nutrient['nutrientName']
            value = nutrient['value']
            row[nutrientName] = value
    rows.append(row)

nutrition_facts_df = pd.DataFrame(rows)


In [ ]:
nutrition_facts_df_pers = nutrition_facts_df['title'].to_frame()

In [ ]:
nutrition_facts_df_pers['fat'] = nutrition_facts_df['Total lipid (fat)'] / 78 * 100
nutrition_facts_df_pers['Saturated fat'] = nutrition_facts_df['Fatty acids, total saturated'] / 20 * 100
nutrition_facts_df_pers['Cholesterol'] = nutrition_facts_df['Cholesterol'] / 300 * 100
nutrition_facts_df_pers['Total carbohydrates'] = nutrition_facts_df['Carbohydrate, by difference'] / 275 * 100
nutrition_facts_df_pers['Sodium'] = nutrition_facts_df['Sodium, Na'] / 2300 * 100
nutrition_facts_df_pers['Dietary Fiber'] = nutrition_facts_df['Fiber, total dietary'] / 28 * 100
nutrition_facts_df_pers['Protein'] = nutrition_facts_df['Protein'] / 50 * 100
nutrition_facts_df_pers['Added sugars'] = nutrition_facts_df['Sugars, added'] / 50 * 100
nutrition_facts_df_pers['Vitamin A'] = nutrition_facts_df['Vitamin A, RAE'] / 900 * 100
nutrition_facts_df_pers['Vitamin C'] = nutrition_facts_df['Vitamin C, total ascorbic acid'] / 90 * 100
nutrition_facts_df_pers['Calcium'] = nutrition_facts_df['Calcium, Ca'] / 1300 * 100
nutrition_facts_df_pers['Iron'] = nutrition_facts_df['Iron, Fe'] / 18 * 100
nutrition_facts_df_pers['Vitamin D'] = nutrition_facts_df['Vitamin D (D2 + D3)'] / 20 * 100
nutrition_facts_df_pers['Vitamin E'] = (nutrition_facts_df['Vitamin E (alpha-tocopherol)'] + nutrition_facts_df['Vitamin E, added']) / 15 * 100
nutrition_facts_df_pers['Vitamin K'] = (nutrition_facts_df['Vitamin K (phylloquinone)'] + nutrition_facts_df['Vitamin K (Dihydrophylloquinone)'] + nutrition_facts_df['Vitamin K (Menaquinone-4)']) / 120 * 100
nutrition_facts_df_pers['Thiamin'] = nutrition_facts_df['Thiamin'] / 1.2 * 100
nutrition_facts_df_pers['Riboflavin'] = nutrition_facts_df['Riboflavin'] / 1.3 * 100
nutrition_facts_df_pers['Niacin'] = nutrition_facts_df['Niacin'] / 16 * 100
nutrition_facts_df_pers['Vitamin B6'] = nutrition_facts_df['Vitamin B-6'] / 1.7 * 100
nutrition_facts_df_pers['Folate'] = nutrition_facts_df['Folate, DFE'] / 400 * 100
nutrition_facts_df_pers['Vitamin B12'] = nutrition_facts_df['Vitamin B-12'] / 2.4 * 100
nutrition_facts_df_pers['Biotin'] = nutrition_facts_df['Biotin'] / 30 * 100
nutrition_facts_df_pers['Pantothenic acid'] = nutrition_facts_df['Pantothenic acid'] / 5 * 100
nutrition_facts_df_pers['Phosphorus'] = nutrition_facts_df['Phosphorus, P'] / 1250 * 100
nutrition_facts_df_pers['Magnesium'] = nutrition_facts_df['Magnesium, Mg'] / 420 * 100
nutrition_facts_df_pers['Zinc'] = nutrition_facts_df['Zinc, Zn'] / 11 * 100
nutrition_facts_df_pers['Selenium'] = nutrition_facts_df['Selenium, Se'] / 55 * 100
nutrition_facts_df_pers['Copper'] = nutrition_facts_df['Copper, Cu'] / 0.9 * 100
nutrition_facts_df_pers['Manganese'] = nutrition_facts_df['Manganese, Mn'] / 2.3 * 100
nutrition_facts_df_pers['Molybdenum'] = nutrition_facts_df['Molybdenum, Mo'] / 45 * 100
nutrition_facts_df_pers['Potassium'] = nutrition_facts_df['Potassium, K'] / 4700 * 100
nutrition_facts_df_pers['Choline'] = nutrition_facts_df['Choline, total'] / 550 * 100


In [ ]:
nutrition_facts_df_pers.fillna(0, inplace=True)

In [ ]:
nutrition_facts_df_pers.to_csv("data/nutrition_facts.csv")

## Similar Recipes